# Notebook 3: Model Definition
This notebook defines the Re-ID model inspired by the paper:
- **ResNet50 backbone** (pretrained on ImageNet)
- **Domain classifier head** with Gradient Reversal Layer (from the paper)
- **ID classifier head** for supervised identity loss
- **Feature embedding layer** for triplet loss

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## Part 1: Gradient Reversal Layer (from the Paper)

In [2]:
class GradientReversalFunction(torch.autograd.Function):
    """
    Gradient Reversal Layer (GRL) — directly from the paper (Section III-B).
    Forward pass: identity function (passes input unchanged).
    Backward pass: multiplies gradient by -lambda (reverses gradient sign).
    This forces the backbone to learn domain-INVARIANT features.
    """
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.save_for_backward(torch.tensor(lambda_))
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        lambda_, = ctx.saved_tensors
        return -lambda_ * grad_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

print('GRL defined!')

GRL defined!


## Part 2: Domain Classifier (from the Paper)

In [3]:
class DomainClassifier(nn.Module):
    """
    Binary domain classifier — inspired by paper Section III-B.
    Predicts whether a feature is from source (PRW) or target (CCTV) domain.
    Combined with GRL to encourage domain-invariant feature learning.
    """
    def __init__(self, in_features=2048, hidden=512):
        super().__init__()
        self.grl = GradientReversalLayer(lambda_=1.0)
        self.classifier = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden, 2)   # 2 classes: source=0, target=1
        )

    def forward(self, x):
        x = self.grl(x)
        return self.classifier(x)

print('Domain Classifier defined!')

Domain Classifier defined!


## Part 3: Full Re-ID Model

In [4]:
class ReIDModel(nn.Module):
    """
    Cross-Domain Person Re-ID Model.
    Inspired by Zhang et al. (2025) — Synthetic-to-Real Video Person Re-ID.

    Architecture:
        ResNet50 backbone
            └─ Feature embedding (2048-d)
                ├─ ID Classifier     → cross-entropy loss (supervised)
                └─ Domain Classifier → domain loss with GRL (self-supervised)
    """
    def __init__(self, num_classes, feat_dim=2048):
        super().__init__()
        self.num_classes = num_classes

        # ── Backbone: ResNet50 pretrained on ImageNet ──────────────
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove final FC layer — we use our own heads
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])

        # ── Global Average Pooling ─────────────────────────────────
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── BN + embedding ────────────────────────────────────────
        self.bn    = nn.BatchNorm1d(feat_dim)
        self.bn.bias.requires_grad_(False)  # no shift

        # ── ID Classifier Head ────────────────────────────────────
        self.id_classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

        # ── Domain Classifier Head (with GRL) ─────────────────────
        self.domain_classifier = DomainClassifier(
            in_features=feat_dim, hidden=512
        )

        self._init_weights()

    def _init_weights(self):
        for m in [self.id_classifier, self.domain_classifier.classifier]:
            for layer in m.modules():
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight)
                    if layer.bias is not None:
                        nn.init.constant_(layer.bias, 0)

    def forward(self, x, return_domain=False):
        # Extract features
        feat = self.backbone(x)      # (B, 2048, H, W)
        feat = self.gap(feat)        # (B, 2048, 1, 1)
        feat = feat.view(feat.size(0), -1)  # (B, 2048)
        feat = self.bn(feat)         # BN normalisation

        # ID prediction
        id_logits = self.id_classifier(feat)

        if return_domain:
            domain_logits = self.domain_classifier(feat)
            return feat, id_logits, domain_logits

        return feat, id_logits

    def get_features(self, x):
        """For evaluation only — returns normalised feature vectors."""
        feat, _ = self.forward(x)
        return F.normalize(feat, p=2, dim=1)

print('ReIDModel defined!')

ReIDModel defined!


In [5]:
# Quick model test — load num_classes from saved dataset info
import pickle
with open('dataset_info.pkl', 'rb') as f:
    info = pickle.load(f)
NUM_CLASSES = info['num_classes']
print(f'Number of training identities (classes): {NUM_CLASSES}')

# Instantiate model
model = ReIDModel(num_classes=NUM_CLASSES).to(DEVICE)

# Test forward pass
dummy = torch.randn(4, 3, 256, 128).to(DEVICE)
feat, id_logits, dom_logits = model(dummy, return_domain=True)

print(f'\nForward pass test:')
print(f'  Input shape         : {dummy.shape}')
print(f'  Feature shape       : {feat.shape}')
print(f'  ID logits shape     : {id_logits.shape}')
print(f'  Domain logits shape : {dom_logits.shape}')
print('\n✅ Model working correctly!')

Number of training identities (classes): 465

Forward pass test:
  Input shape         : torch.Size([4, 3, 256, 128])
  Feature shape       : torch.Size([4, 2048])
  ID logits shape     : torch.Size([4, 465])
  Domain logits shape : torch.Size([4, 2])

✅ Model working correctly!


In [6]:
# Count model parameters
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}')
print(f'\n✅ Ready for Notebook 4: Training')

Total parameters     : 25,849,875
Trainable parameters : 25,847,827

✅ Ready for Notebook 4: Training
